### Notebook 5 — Feature Engineering
##### Goal
FAISS gives us candidate items.

Now we calculate features for each candidate:

| Feature          | Meaning                               |
| ---------------- | ------------------------------------- |
| similarity_score | User-item similarity                  |
| popularity_score | How popular the item is               |
| recency_score    | How recent the item is                |
| preference_score | How much the user likes this category |
| label            | Ground truth for training             |


### Import Required libraries

In [15]:
import pandas as pd
import numpy as np
import faiss
import pickle
from sklearn.metrics.pairwise import cosine_similarity

### load Files

In [17]:
items_df = pd.read_csv("data/item_df.csv")

with open("data/item_embeddings.pkl", "rb") as f:
    item_embeddings = pickle.load(f)

with open("data/user_embeddings.pkl", "rb") as f:
    user_embeddings = pickle.load(f)

Load embeddings:

In [18]:
item_embeddings = np.array(item_embeddings).astype("float32")

dimension = item_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(item_embeddings)

In [19]:
user_vector = np.array(
    user_embeddings[1]
).astype("float32")

user_vector = user_vector.reshape(1, -1)

In [20]:
D, I = index.search(
    user_vector,
    k=10
)

In [21]:
recommended_items = items_df.iloc[I[0]]

recommended_items

,item_id,title,category
658,659,Content 659,Business
358,359,Content 359,Machine Learning
357,358,Content 358,LLM
638,639,Content 639,Deep Learning
758,759,Content 759,Finance
634,635,Content 635,LLM
369,370,Content 370,Machine Learning
395,396,Content 396,Computer Vision
767,768,Content 768,Deep Learning
768,769,Content 769,Python


### Create Item Embedding Dictionary

In [22]:
item_embedding_dict = {}

for idx, row in items_df.iterrows():

    item_embedding_dict[row["item_id"]] = item_embeddings[idx]

### Popularity Feature
Count interactions per item.

In [23]:
popularity = (
    interactions_df
    .groupby("item_id")
    .size()
)

In [24]:
popularity.head()

item_id
1     9
2    11
3    18
4    11
5     8
dtype: int64

### Choose User

In [11]:
user_id = 1

user_vector = user_embeddings[user_id]

### Candidate Items
Use the same recommendations from Notebook 4:

In [25]:
candidate_items = recommended_items.copy()

### Create Feature List

In [26]:
features = []

### Generate Features

In [30]:
for _, row in candidate_items.iterrows():

    item_id = row["item_id"]

    item_vector = item_embedding_dict[item_id]

    similarity_score = cosine_similarity(
        user_vector.reshape(1,-1),
        item_vector.reshape(1,-1)
    )[0][0]

    popularity_score = popularity.get(
        item_id,
        0
    )

    recency_score = np.random.randint(
        1,
        30
    )

    preference_score = np.random.uniform(
        0.5,
        1.0
    )

    label = np.random.randint(
        0,
        2
    )

    features.append([
        user_id,
        item_id,
        similarity_score,
        popularity_score,
        recency_score,
        preference_score,
        label
    ])

### Create DataFrame

In [31]:
feature_df = pd.DataFrame(
    features,
    columns=[
        "user_id",
        "item_id",
        "similarity_score",
        "popularity_score",
        "recency_score",
        "preference_score",
        "label"
    ]
)

### View Features

In [32]:
feature_df.head()

,user_id,item_id,similarity_score,popularity_score,recency_score,preference_score,label
0,1,659,0.918997,13,22,0.513196,0
1,1,359,0.895065,12,9,0.786582,0
2,1,358,0.893963,11,16,0.798852,0
3,1,639,0.892239,9,21,0.868703,1
4,1,759,0.890143,8,8,0.712731,0


### Save Dataset

In [33]:
feature_df.to_csv(
    "data/ranking_features.csv",
    index=False
)

### Important Note
For now:

In [35]:
recency_score = np.random.randint(1, 30)

preference_score = np.random.uniform(0.5, 1.0)

label = np.random.randint(0, 2)

Inside your loop, it should look like:

In [36]:
for _, row in candidate_items.iterrows():

    item_id = row["item_id"]

    item_vector = item_embedding_dict[item_id]

    similarity_score = cosine_similarity(
        user_vector.reshape(1,-1),
        item_vector.reshape(1,-1)
    )[0][0]

    popularity_score = popularity.get(item_id, 0)

    recency_score = np.random.randint(1, 30)

    preference_score = np.random.uniform(0.5, 1.0)

    label = np.random.randint(0, 2)

    features.append([
        user_id,
        item_id,
        similarity_score,
        popularity_score,
        recency_score,
        preference_score,
        label
    ])

In [37]:
feature_df = pd.DataFrame(
    features,
    columns=[
        "user_id",
        "item_id",
        "similarity_score",
        "popularity_score",
        "recency_score",
        "preference_score",
        "label"
    ]
)

feature_df.head()

,user_id,item_id,similarity_score,popularity_score,recency_score,preference_score,label
0,1,659,0.918997,13,22,0.513196,0
1,1,359,0.895065,12,9,0.786582,0
2,1,358,0.893963,11,16,0.798852,0
3,1,639,0.892239,9,21,0.868703,1
4,1,759,0.890143,8,8,0.712731,0
